# Trabajo 1: Optimización Numérica y Metaheurística
**Materia:** Redes Neuronales y Algoritmos Bioinspirados  
**Universidad Nacional de Colombia**

---

### Descripción del Notebook
Este cuaderno de trabajo está estructurado como un entorno de experimentación riguroso y reproducible, dividido en dos secciones principales:

1. **Sección 1: Demostración y Análisis de Funciones y Algoritmos**
   - Formulación matemática rigurosa de las funciones objetivo de prueba (**Rosenbrock** y **Rastrigin**).
   - Explicación conceptual de cada uno de los 5 métodos de optimización implementados (Descenso por Gradiente Proyectado con Armijo, Algoritmo Genético Real, Enjambre de Partículas PSO, Evolución Diferencial, Búsqueda Aleatoria).
   - Demostraciones prácticas con la función modular `optimize()`, utilizando el parámetro `plot=True` para generar **animaciones interactivas en Plotly** tanto en 2D (con curvas de nivel) como en 3D (en el espacio de búsqueda tridimensional).

2. **Sección 2: Comparación Rigurosa y Búsqueda de Hiperparámetros**
   - **Justificación del conteo de evaluaciones equivalentes:** Justificación matemática de por qué el costo de aproximar el gradiente mediante diferencias finitas centrales en dimensión $d$ es exactamente $2d \cdot n_{\text{grad}}$ evaluaciones de $f$, estableciendo la fórmula de equivalencia $n_{\text{equiv}} = n_f + 2d \cdot n_{\text{grad}}$ y la necesidad de un presupuesto común para comparaciones justas.
   - **Sintonización de hiperparámetros (Grid Search):** Búsqueda sistemática sobre una rejilla de hiperparámetros clave para identificar las mejores configuraciones de cada método.
   - **Experimento estadístico (30 corridas independientes):** Ejecución de 30 corridas independientes por configuración para Rosenbrock y Rastrigin (en 2D y 3D), evaluando media, desviación estándar, mediana, mejor y peor valor, tasa de éxito bajo umbrales formales, y visualización con boxplots y curvas de convergencia.
**Corrección para la sustentación:** modifique `X0_2D` y `X0_3D` en la primera celda para ejecutar GD desde el punto solicitado. Se mantienen Rosenbrock, Rastrigin, Plotly y el protocolo experimental del notebook original. Se retiraron las salidas antiguas para evitar confundirlas con una corrida corregida.

### Funciones de prueba y condición inicial

Para $d\in\{2,3\}$:

$$f_{\mathrm{Rosenbrock}}(\mathbf{x})=\sum_{i=1}^{d-1}\left[100(x_{i+1}-x_i^2)^2+(1-x_i)^2\right],\quad \mathbf{x}\in[-2.048,2.048]^d.$$

$$f_{\mathrm{Rastrigin}}(\mathbf{x})=10d+\sum_{i=1}^d\left[x_i^2-10\cos(2\pi x_i)\right],\quad \mathbf{x}\in[-5.12,5.12]^d.$$

Los mínimos globales son $f(\mathbf{1})=0$ para Rosenbrock y $f(\mathbf{0})=0$ para Rastrigin. El punto inicial $\mathbf{x}^{(0)}$ no es el óptimo conocido: es una entrada del experimento. En la primera celda se puede cambiar sin editar la implementación del algoritmo. Si queda fuera del dominio o su dimensión no coincide, se rechaza con un mensaje explícito.

Los comandos de sustentación también admiten `python punto_1/funcion1.py run --algo gradient_descent --dim 2 --x0 -1.2 1`. Los hiperparámetros del módulo del notebook usan nombres en español, por ejemplo `tasa_aprendizaje_inicial` y `factor_reduccion`.


In [ ]:
# Cargar exactamente el módulo que acompaña a este notebook.
from pathlib import Path
import sys
import importlib.util
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'punto_1/Experimentos.ipynb').exists()), None)
if RAIZ is None:
    raise FileNotFoundError('Abra el notebook dentro del repositorio')
spec = importlib.util.spec_from_file_location('funcion1_notebook', RAIZ/'punto_1/funcion1.py')
modulo = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = modulo
spec.loader.exec_module(modulo)
optimize = modulo.optimize
graficar_optimizacion = modulo.graficar_optimizacion
ALGORITHMS, PROBLEMS = modulo.ALGORITHMS, modulo.PROBLEMS
run_batch, MAIN_ALGOS = modulo.run_batch, modulo.MAIN_ALGOS
SALIDA = RAIZ/'results/parte1_notebook_corregida'
SALIDA.mkdir(parents=True, exist_ok=True)

# Condiciones iniciales editables para la sustentación; respetar el dominio.
X0_2D = [-1.2, 1.0]
X0_3D = [-1.2, 1.0, 0.5]
PRESUPUESTO_DEMO = 5000

def publicar_figura(fig, nombre):
    # HTML autónomo: conserva controles y animación para incrustar en el blog.
    fig.write_html(SALIDA/f'{nombre}.html', include_plotlyjs=True, auto_play=False)
    fig.show()

print('Módulo cargado:', modulo.__file__)
print('Algoritmos:', sorted(ALGORITHMS))


In [ ]:
# Verifica la condición inicial exacta sin gastar el presupuesto de una corrida.
for dim, punto in [(2, X0_2D), (3, X0_3D)]:
    verificacion = optimize('gradient_descent', 'rosenbrock', dim=dim,
                            x0=punto, max_evals=1, record=True)
    np.testing.assert_array_equal(verificacion.frames[0], punto)
    print(f'GD {dim}D comienza exactamente en {verificacion.frames[0]}')


### 1.2 Algoritmos de Optimización Implementados

El módulo implementa cuatro familias principales y dos líneas base de referencia:

1. **Descenso por Gradiente Proyectado (Gradient Descent - GD) con Armijo:**
   - Calcula el gradiente numérico por diferencias finitas centrales:
     $$\nabla f(x)_i \approx \frac{f(x + h e_i) - f(x - h e_i)}{2h}, \quad h = 10^{-6}$$
   - Actualización proyectada al dominio: $x_{k+1} = \text{clip}(x_k - \alpha_k \nabla f(x_k), \text{lim\_inf}, \text{lim\_sup})$.
   - Condición de retroceso de Armijo proyectada de Bertsekas:
     $$f(x_{k+1}) \le f(x_k) - c \, \nabla f(x_k)^T (x_k - x_{k+1})$$
     reduciendo $\alpha \leftarrow \text{factor\_reduccion} \cdot \alpha$ progresivamente.

2. **Algoritmo Genético de Codificación Real (Genetic Algorithm - GA):**
   - Selección por torneo entre $k$ individuos de la población.
   - Cruce BLX-$\alpha$: genera descendencia muestreando uniformemente en el intervalo expandido $[\min(p_1, p_2) - \alpha d, \max(p_1, p_2) + \alpha d]$.
   - Mutación gaussiana con decaimiento geométrico de la escala $\sigma$:
     $$\sigma = \sigma_0 \cdot \text{rango} \cdot \left(\frac{\sigma_{\text{final}}}{\sigma_0}\right)^{\frac{n_{\text{equiv}}}{max\_evals}}$$
   - Elitismo que preserva intactos a los mejores individuos de cada generación.

3. **Optimización por Enjambre de Partículas (PSO):**
   - Sistema multiagente donde cada partícula actualiza su velocidad y posición usando factores de constricción canónicos de Clerc-Kennedy ($w=0.7298, c_1=c_2=1.49618$):
     $$v_i^{k+1} = w v_i^k + c_1 r_1 (p_{\text{best}, i} - x_i^k) + c_2 r_2 (g_{\text{best}} - x_i^k)$$
     $$x_i^{k+1} = \text{clip}(x_i^k + v_i^{k+1}, \text{lim\_inf}, \text{lim\_sup})$$
   - Manejo de frontera por absorción de velocidad ($v=0$ al impactar los límites).

4. **Evolución Diferencial (Differential Evolution - DE, esquema rand/1/bin):**
   - Generación de vectores mutantes mediante diferencias vectoriales: $v = x_a + F (x_b - x_c)$ con índices mutuamente disjuntos.
   - Cruce binomial con probabilidad $CR$, garantizando al menos un gen mutante.
   - Selección voraz $1$ a $1$: el vector de prueba reemplaza al individuo si y solo si su aptitud es menor o igual.

5. **Líneas Base de Comparación:**
   - **Búsqueda Aleatoria (Random Search):** Muestreo uniforme puro en el espacio.

In [ ]:
# Demostración 1: GD en Rosenbrock 2D, desde el punto solicitado.
res_gd_2d = optimize('gradient_descent', 'rosenbrock', dim=2,
                     x0=X0_2D, random_seed=42,
                     max_evals=PRESUPUESTO_DEMO, record=True)
print(res_gd_2d)
publicar_figura(graficar_optimizacion(res_gd_2d), 'figura_1_gd_rosenbrock_2d')


In [ ]:
# =====================================================================
# Visualización con Matplotlib (Fondo blanco) - Trayectoria y Convergencia
# =====================================================================
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor='white')

# 1. Contornos de Rosenbrock 2D y trayectoria del algoritmo
x_vals = np.linspace(-2.048, 2.048, 300)
y_vals = np.linspace(-2.048, 2.048, 300)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
Z_grid = 100.0 * (Y_grid - X_grid**2)**2 + (1.0 - X_grid)**2

niveles = np.logspace(0, 3.5, 25)
cs = ax1.contour(X_grid, Y_grid, Z_grid, levels=niveles, cmap='viridis', alpha=0.85)
ax1.clabel(cs, inline=True, fontsize=7, fmt='%.0f')

trayectoria = np.array(res_gd_2d.frames)
ax1.plot(trayectoria[:, 0], trayectoria[:, 1], color='#e74c3c', linewidth=2.0,
         marker='o', markersize=3, label='Trayectoria GD')
ax1.plot(trayectoria[0, 0], trayectoria[0, 1], marker='s', markersize=8,
         color='#2980b9', label=f'Inicio ({trayectoria[0, 0]:.2f}, {trayectoria[0, 1]:.2f})')
ax1.plot(1.0, 1.0, marker='*', markersize=14, color='#f1c40f',
         markeredgecolor='black', label='Óptimo global (1, 1)')

ax1.set_title('Descenso por Gradiente en Rosenbrock 2D', fontsize=12, fontweight='bold', color='black')
ax1.set_xlabel(r'$x_1$', fontsize=11, color='black')
ax1.set_ylabel(r'$x_2$', fontsize=11, color='black')
ax1.set_xlim(-2.048, 2.048)
ax1.set_ylim(-2.048, 2.048)
ax1.grid(True, linestyle='--', alpha=0.5, color='gray')
ax1.legend(loc='upper left', framealpha=0.95, facecolor='white', edgecolor='lightgray')
ax1.tick_params(colors='black')

# 2. Curva de convergencia semilogarítmica
ax2.semilogy(res_gd_2d.eval_trace, res_gd_2d.history, color='#2c3e50', linewidth=2, label='f(x) acumulado')
ax2.axhline(0.0, color='gray', linestyle=':', label='Mínimo global f* = 0')
ax2.set_title('Convergencia vs. Evaluaciones Equivalentes', fontsize=12, fontweight='bold', color='black')
ax2.set_xlabel(r'Evaluaciones equivalentes ($n_f + 2d n_{grad}$)', fontsize=11, color='black')
ax2.set_ylabel(r'$f(x)$ [escala log]', fontsize=11, color='black')
ax2.grid(True, which='both', linestyle='--', alpha=0.5, color='gray')
ax2.legend(loc='upper right', framealpha=0.95, facecolor='white', edgecolor='lightgray')
ax2.tick_params(colors='black')

fig.text(.5,.01,'Fuente: elaboración propia; punto_1/funcion1.py; condición inicial configurable.',ha='center',fontsize=8)
plt.tight_layout(rect=(0,.04,1,1))
fig.savefig(SALIDA/'figura_2_trayectoria_convergencia.png',dpi=160)
plt.show()


In [ ]:
res_pso_2d = optimize('pso', 'rastrigin', dim=2, random_seed=42,
                      max_evals=PRESUPUESTO_DEMO, record=True)
print(res_pso_2d)
publicar_figura(graficar_optimizacion(res_pso_2d), 'figura_3_pso_rastrigin_2d')


In [ ]:
# Demostración 3: DE sobre Rosenbrock, realmente en 3D.
res_de_3d = optimize('differential_evolution', 'rosenbrock', dim=3,
                     random_seed=42, max_evals=PRESUPUESTO_DEMO, record=True)
print(res_de_3d)
publicar_figura(graficar_optimizacion(res_de_3d), 'figura_4_de_rosenbrock_3d')


In [ ]:
# Demostración 4: GD sobre Rosenbrock, realmente en 3D y con x0 explícito.
res_gd_3d = optimize('gradient_descent', 'rosenbrock', dim=3,
                     x0=X0_3D, random_seed=42,
                     max_evals=PRESUPUESTO_DEMO, record=True)
print(res_gd_3d)
publicar_figura(graficar_optimizacion(res_gd_3d), 'figura_5_gd_rosenbrock_3d')


---
## Sección 2: Comparación Rigurosa y Ajuste de Hiperparámetros

### 2.1 Justificación Teórica del Conteo de Evaluaciones Equivalentes

En la literatura de optimización numérica y metaheurística, comparar algoritmos basándose únicamente en el número de "iteraciones" o "épocas" constituye un sesgo metodológico grave:

1. **Costo de un algoritmo poblacional:**
   En cada iteración, un algoritmo genético o enjambre con tamaño de población $N$ evalúa la función objetivo exactamente $N$ veces.

2. **Costo de un algoritmo basado en gradiente numérico:**
   Calcular el gradiente numérico $\nabla f(x)$ en $\mathbb{R}^d$ mediante diferencias finitas centrales requiere perturbar la función hacia adelante y hacia atrás en cada una de las $d$ direcciones canónicas:
   $$\nabla f(x)_i \approx \frac{f(x + h e_i) - f(x - h e_i)}{2h}, \quad i = 1, \dots, d$$
   Esto representa exactamente **$2d$ evaluaciones de la función objetivo** por cada gradiente computado. Además, el procedimiento de retroceso de Armijo realiza evaluaciones adicionales de $f$ para validar el tamaño de paso.

#### Fórmula de Equivalencia Justificada:
$$n_{\text{equiv}} = n_f + 2d \cdot n_{\text{grad}}$$

* Para algoritmos metaheurísticos libres de derivadas ($n_{\text{grad}} = 0$): $n_{\text{equiv}} = n_f$.
* Para Descenso por Gradiente: Cada llamada al gradiente suma $2d$ evaluaciones al contador equivalente.
* **Presupuesto Fijo ($max\_evals$):** Todos los métodos son sometidos exactamente al mismo límite superior de evaluaciones equivalentes de la caja negra.

### 2.2 Sintonización de Hiperparámetros (Grid Search)

A continuación, implementamos una búsqueda sistemática en rejilla (**Grid Search**) para encontrar la combinación óptima de hiperparámetros de cada algoritmo sobre una muestra de semillas independientes ($seed \in \{10, 20, 30\}$).

Los hiperparámetros a explorar son:
* **Gradient Descent (GD):** `tasa_aprendizaje_inicial` $\in [0.1, 0.5, 1.0]$, `factor_reduccion` $\in [0.3, 0.5]$.
* **PSO:** `inercia` $\in [0.5, 0.7298, 0.9]$, `num_particulas` $\in [20, 30, 40]$.
* **Differential Evolution (DE):** `factor_mutacion` $\in [0.5, 0.8]$, `tasa_cruce` $\in [0.7, 0.9]$, `tam_poblacion` $\in [30, 40]$.
* **Genetic Algorithm (GA):** `tam_poblacion` $\in [30, 50]$, `prob_cruce` $\in [0.8, 0.95]$, `alpha_blx` $\in [0.3, 0.5]$.

In [ ]:
import itertools
# Función modular para realizar Grid Search sobre un algoritmo dado
def grid_search_algorithm(algo_name, problem="rosenbrock", dim=2, param_grid=None, seeds=(10, 20, 30), max_evals=5000):
    """
    Evalúa todas las combinaciones de hiperparámetros sobre un conjunto de semillas
    y retorna la configuración con menor mediana de la función objetivo.
    """
    if param_grid is None or len(param_grid) == 0:
        return {}, 0.0

    keys = list(param_grid.keys())
    values = list(param_grid.values())
    combinaciones = [dict(zip(keys, prod)) for prod in itertools.product(*values)]
    
    resultados_grid = []
    
    for params in combinaciones:
        f_vals = []
        for s in seeds:
            res = optimize(algo_name, problem=problem, dim=dim, random_seed=s, max_evals=max_evals, **params)
            f_vals.append(res.f_best)
        
        mediana_f = float(np.median(f_vals))
        media_f = float(np.mean(f_vals))
        resultados_grid.append({
            "algoritmo": algo_name,
            "parametros": params,
            "mediana_f": mediana_f,
            "media_f": media_f
        })
    
    df_grid = pd.DataFrame(resultados_grid).sort_values(by="mediana_f")
    mejor_config = df_grid.iloc[0]["parametros"]
    mejor_mediana = df_grid.iloc[0]["mediana_f"]
    
    return mejor_config, df_grid

# Rejillas de búsqueda
grid_gd = {
    "tasa_aprendizaje_inicial": [0.1, 0.5, 1.0],
    "factor_reduccion": [0.3, 0.5]
}

grid_pso = {
    "inercia": [0.5, 0.7298, 0.9],
    "num_particulas": [20, 30, 40]
}

grid_de = {
    "factor_mutacion": [0.5, 0.8],
    "tasa_cruce": [0.7, 0.9],
    "tam_poblacion": [30, 40]
}

grid_ga = {
    "tam_poblacion": [30, 50],
    "prob_cruce": [0.8, 0.95],
    "alpha_blx": [0.3, 0.5]
}

print("Iniciando Grid Search para Rosenbrock 2D...")
mejores_params = {}

for nombre_algo, rejilla in [
    ("gradient_descent", grid_gd),
    ("pso", grid_pso),
    ("differential_evolution", grid_de),
    ("genetic", grid_ga)
]:
    best_cfg, df_res = grid_search_algorithm(nombre_algo, "rosenbrock", 2, rejilla, seeds=(10, 20, 30), max_evals=4000)
    mejores_params[nombre_algo] = best_cfg
    print(f" -> {nombre_algo:22s} | Mejor configuración: {best_cfg}")

print("\n Hiperparámetros optimizados con éxito.")

### 2.3 Experimento Estadístico: 30 Corridas Independientes por Configuración

Con los mejores hiperparámetros calibrados, procedemos a realizar la comparación rigurosa con **al menos 30 corridas independientes** ($seed = 0, 1, \dots, 29$) para cada algoritmo en las siguientes 4 configuraciones:
1. **Rosenbrock 2D**
2. **Rosenbrock 3D**
3. **Rastrigin 2D**
4. **Rastrigin 3D**

#### Criterio de Éxito Formal:
* Para **Rosenbrock**: Se considera éxito si la solución final alcanza $f_{\text{best}} < 10^{-4}$ (muy cercano al óptimo global en $(1, \dots, 1)$).
* Para **Rastrigin**: Se considera éxito si la solución final alcanza $f_{\text{best}} < 10^{-2}$ (evadiendo todos los mínimos locales hacia $(0, \dots, 0)$).

In [ ]:
# Ejecución del experimento estadístico formal sobre 30 corridas independientes
N_RUNS = 30
SEED0 = 0
MAX_EVALS = 10000

algoritmos_a_evaluar = list(MAIN_ALGOS) + ["random_search"]
escenarios = [
    ("rosenbrock", 2, 1e-4),
    ("rosenbrock", 3, 1e-4),
    ("rastrigin", 2, 1e-2),
    ("rastrigin", 3, 1e-2)
]

tablas_resumen = {}
datos_completos = []

print(f"Iniciando evaluación estadística ({N_RUNS} corridas por configuración, presupuesto={MAX_EVALS} evals equiv)...\n")

for prob, d, umbral in escenarios:
    filas_escenario = []
    t_inicio = time.time()
    
    for algo in algoritmos_a_evaluar:
        params_algo = mejores_params.get(algo, {})
        stats = run_batch(algo, prob, d, N_RUNS, SEED0, MAX_EVALS, umbral, params_algo)
        filas_escenario.append(stats)
        
        # Guardar corridas individuales para visualización de distribuciones
        for i in range(N_RUNS):
            res_ind = optimize(algo, prob, d, SEED0 + i, MAX_EVALS, **params_algo)
            datos_completos.append({
                "problema": prob,
                "dimension": d,
                "algoritmo": algo,
                "corrida": i,
                "f_best": res_ind.f_best,
                "log10_f": np.log10(max(res_ind.f_best, 1e-30)),
                "n_f": res_ind.n_f,
                "n_grad": res_ind.n_grad,
                "n_equiv": res_ind.n_equiv
            })
            
    df_escenario = pd.DataFrame(filas_escenario)
    tablas_resumen[f"{prob}_{d}D"] = df_escenario
    duracion = time.time() - t_inicio
    print(f"Completado {prob.capitalize()} {d}D en {duracion:.2f} s.")

df_todas_las_corridas = pd.DataFrame(datos_completos)
print("\n Todas las corridas estadísticas han finalizado con éxito.")

In [ ]:
# Mostrar y formatear tablas de resultados estadísticos
columnas_formato = {
    "algorithm": "Algoritmo",
    "mean": "Media (f)",
    "std": "Desv. Est. (f)",
    "median": "Mediana (f)",
    "best": "Mejor (f_min)",
    "worst": "Peor (f_max)",
    "success_rate": "Tasa Éxito (%)",
    "mean_n_f": "Media n_f",
    "mean_n_grad": "Media n_grad",
    "mean_n_equiv": "Media n_equiv"
}

for clave, df in tablas_resumen.items():
    print(f"\n{'='*75}")
    print(f"RESULTADOS ESTADÍSTICOS: {clave.upper()} (30 Corridas)")
    print(f"{'='*75}")
    
    df_mostrar = df[list(columnas_formato.keys())].rename(columns=columnas_formato).copy()
    
    # Formatear números para presentación científica
    for col in ["Media (f)", "Desv. Est. (f)", "Mediana (f)", "Mejor (f_min)", "Peor (f_max)"]:
        df_mostrar[col] = df_mostrar[col].apply(lambda v: f"{v:.3e}")
    df_mostrar["Tasa Éxito (%)"] = df_mostrar["Tasa Éxito (%)"].apply(lambda v: f"{v*100:.1f}%")
    df_mostrar["Media n_f"] = df_mostrar["Media n_f"].apply(lambda v: f"{v:.0f}")
    df_mostrar["Media n_grad"] = df_mostrar["Media n_grad"].apply(lambda v: f"{v:.0f}")
    df_mostrar["Media n_equiv"] = df_mostrar["Media n_equiv"].apply(lambda v: f"{v:.0f}")
    
    display(df_mostrar)

In [ ]:
# Visualización 1: Distribución del Desempeño (Boxplots de log10(f_best))
fig_box = px.box(
    df_todas_las_corridas,
    x="algoritmo",
    y="log10_f",
    color="algoritmo",
    facet_row="problema",
    facet_col="dimension",
    title="Comparación de Distribución de Resultados: log₁₀(f_best) en 30 Corridas",
    labels={"log10_f": "log₁₀(f_best)", "algoritmo": "Algoritmo"},
    template="plotly_dark",
    height=750
)
fig_box.update_layout(showlegend=False)
fig_box.add_annotation(text='Fuente: elaboración propia; Experimentos.ipynb.', xref='paper', yref='paper', x=0, y=-0.15, showarrow=False)
publicar_figura(fig_box, 'figura_15_comparacion')

In [ ]:
# =====================================================================
# Visualización Matplotlib (Fondo blanco) - Boxplots con Línea de Óptimo
# =====================================================================
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='white')

# Umbral de éxito / tolerancia para representar el óptimo en escala logarítmica
LOG_OPTIMO_TOL = -4.0  # Corresponde a f(x) <= 1e-4

configuraciones_box = [
    (('rosenbrock', 2), axes[0, 0], 'Rosenbrock 2D'),
    (('rosenbrock', 3), axes[0, 1], 'Rosenbrock 3D'),
    (('rastrigin', 2), axes[1, 0], 'Rastrigin 2D'),
    (('rastrigin', 3), axes[1, 1], 'Rastrigin 3D')
]

colores_paleta = {
    'gradient_descent': '#EF553B',
    'pso': '#00CC96',
    'differential_evolution': '#636EFA',
    'genetic': '#AB63FA',
    'random_search': '#FFA15A',
}

for (prob, d), ax, titulo in configuraciones_box:
    sub_df = df_todas_las_corridas[
        (df_todas_las_corridas['problema'] == prob) & 
        (df_todas_las_corridas['dimension'] == d)
    ]
    algos_presentes = [a for a in algoritmos_a_evaluar if a in sub_df['algoritmo'].unique()]
    
    data_plot = [sub_df[sub_df['algoritmo'] == a]['log10_f'].values for a in algos_presentes]
    etiquetas = [a.replace('_', ' ').title() for a in algos_presentes]
    
    bplot = ax.boxplot(
        data_plot,
        labels=etiquetas,
        patch_artist=True,
        notch=False,
        medianprops=dict(color='black', linewidth=1.8),
        flierprops=dict(marker='o', markersize=4, alpha=0.6, markeredgecolor='gray')
    )
    
    for patch, algo in zip(bplot['boxes'], algos_presentes):
        c = colores_paleta.get(algo, '#3498db')
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
    
    # --- Línea horizontal de referencia del Óptimo Global ---
    # Usamos \leq (compatible con Matplotlib) o <=
    linea_opt = ax.axhline(
        y=LOG_OPTIMO_TOL,
        color='#d63031',
        linestyle='--',
        linewidth=1.8,
        label=r'Umbral de referencia $10^{-4}$; no es el óptimo'
    )
    
    # Ajustar límite inferior del eje Y para que siempre se vea la línea del óptimo
    y_min_data = min([vals.min() for vals in data_plot if len(vals) > 0], default=LOG_OPTIMO_TOL)
    ax.set_ylim(bottom=min(y_min_data - 0.5, LOG_OPTIMO_TOL - 1.0))
    
    ax.set_title(f'Distribución de Desempeño: {titulo}', fontsize=12, fontweight='bold', color='black')
    ax.set_ylabel(r'$\log_{10}(f_{\mathrm{best}})$', fontsize=11, color='black')
    ax.grid(True, linestyle='--', alpha=0.5, color='gray')
    ax.tick_params(colors='black', labelsize=9)
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
    
    # Leyenda solo en el primer cuadrante para no saturar
    if ax is axes[0, 0]:
        ax.legend(loc='upper right', framealpha=0.95, facecolor='white', edgecolor='lightgray', fontsize=9)

plt.suptitle('Comparación Estadística de Desempeño (30 Corridas) vs. Óptimo Teórico', 
             fontsize=14, fontweight='bold', y=0.995, color='black')
fig.text(.5,.01,'Fuente: elaboración propia; Experimentos.ipynb.',ha='center',fontsize=8)
plt.tight_layout(rect=(0,.04,1,1))
fig.savefig(SALIDA/'figura_16_comparacion.png',dpi=160)
plt.show()

In [ ]:
# Visualización 2: Curvas de Convergencia Promedio a lo largo de las Evaluaciones
fig_conv = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Rosenbrock 2D", "Rosenbrock 3D", "Rastrigin 2D", "Rastrigin 3D"),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

posiciones_sub = {
    ("rosenbrock", 2): (1, 1),
    ("rosenbrock", 3): (1, 2),
    ("rastrigin", 2): (2, 1),
    ("rastrigin", 3): (2, 2),
}

colores_algo = {
    "gradient_descent": "#EF553B",
    "pso": "#00CC96",
    "differential_evolution": "#636EFA",
    "genetic": "#AB63FA",
    "random_search": "#FFA15A",
}

for (prob, d), (row, col) in posiciones_sub.items():
    for algo in algoritmos_a_evaluar:
        params_algo = mejores_params.get(algo, {})
        # Promediar historial sobre 5 semillas representativas
        historiales = []
        eval_traces = []
        for s in range(5):
            r = optimize(algo, prob, d, s, MAX_EVALS, **params_algo)
            historiales.append(r.history)
            eval_traces.append(r.eval_trace)
        
        # Graficar la trayectoria media
        min_len = min(len(h) for h in historiales)
        arr_f = np.array([h[:min_len] for h in historiales])
        arr_ev = np.array([e[:min_len] for e in eval_traces])
        
        media_f = np.mean(arr_f, axis=0)
        media_ev = np.mean(arr_ev, axis=0)
        
        fig_conv.add_trace(
            go.Scatter(
                x=media_ev,
                y=np.log10(np.maximum(media_f, 1e-30)),
                mode="lines",
                name=algo,
                line=dict(color=colores_algo[algo], width=2),
                showlegend=(row == 1 and col == 1)
            ),
            row=row, col=col
        )
        fig_conv.update_xaxes(title_text="Evals Equivalentes", row=row, col=col)
        fig_conv.update_yaxes(title_text="log₁₀(f_best)", row=row, col=col)

fig_conv.update_layout(
    title_text="Curvas de Convergencia Promedio vs Presupuesto Equivalente",
    template="plotly_dark",
    height=750,
    width=950
)
fig_conv.add_annotation(text='Fuente: elaboración propia; Experimentos.ipynb.', xref='paper', yref='paper', x=0, y=-0.15, showarrow=False)
publicar_figura(fig_conv, 'figura_17_comparacion')

In [ ]:
# =====================================================================
# Visualización Matplotlib (Fondo blanco) - Curvas de Convergencia
# =====================================================================
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

fig, axes = plt.subplots(2, 2, figsize=(15, 10), facecolor='white')

posiciones_mpl = [
    (('rosenbrock', 2), axes[0, 0], 'Rosenbrock 2D'),
    (('rosenbrock', 3), axes[0, 1], 'Rosenbrock 3D'),
    (('rastrigin', 2), axes[1, 0], 'Rastrigin 2D'),
    (('rastrigin', 3), axes[1, 1], 'Rastrigin 3D')
]

colores_algo_mpl = {
    'gradient_descent': '#EF553B',
    'pso': '#00CC96',
    'differential_evolution': '#636EFA',
    'genetic': '#AB63FA',
    'random_search': '#FFA15A',
}

for (prob, d), ax, titulo in posiciones_mpl:
    for algo in algoritmos_a_evaluar:
        params_algo = mejores_params.get(algo, {})
        historiales = []
        eval_traces = []
        for s in range(5):
            r = optimize(algo, prob, d, s, MAX_EVALS, **params_algo)
            historiales.append(r.history)
            eval_traces.append(r.eval_trace)
        
        min_len = min(len(h) for h in historiales)
        arr_f = np.array([h[:min_len] for h in historiales])
        arr_ev = np.array([e[:min_len] for e in eval_traces])
        
        media_f = np.mean(arr_f, axis=0)
        media_ev = np.mean(arr_ev, axis=0)
        
        ax.plot(
            media_ev,
            np.log10(np.maximum(media_f, 1e-30)),
            label=algo.replace('_', ' ').title(),
            color=colores_algo_mpl.get(algo, '#333333'),
            linewidth=2.0
        )
        
    ax.set_title(f'Convergencia Promedio: {titulo}', fontsize=12, fontweight='bold', color='black')
    ax.set_xlabel(r'Evaluaciones Equivalentes ($n_f + 2d \cdot n_{\mathrm{grad}}$)', fontsize=10, color='black')
    ax.set_ylabel(r'$\log_{10}(f_{\mathrm{best}})$', fontsize=10, color='black')
    ax.grid(True, linestyle='--', alpha=0.5, color='gray')
    ax.tick_params(colors='black')
    if ax is axes[0, 0]:
        ax.legend(loc='upper right', framealpha=0.95, facecolor='white', edgecolor='lightgray', fontsize=9)

plt.suptitle('Curvas de Convergencia Promedio vs. Presupuesto Computacional Equivalente', fontsize=14, fontweight='bold', y=0.995, color='black')
fig.text(.5,.01,'Fuente: elaboración propia; Experimentos.ipynb.',ha='center',fontsize=8)
plt.tight_layout(rect=(0,.04,1,1))
fig.savefig(SALIDA/'figura_18_comparacion.png',dpi=160)
plt.show()


### 2.4 Interpretación de los resultados de esta ejecución

Las conclusiones numéricas se obtienen de las tablas anteriores; no se conservan porcentajes históricos como si correspondieran a una nueva corrida. Compare media, desviación y tasa de éxito para cada función y dimensión, sin generalizar un ganador fuera de estos problemas y presupuestos.

La búsqueda se mantiene calibrada en Rosenbrock 2D con las semillas declaradas: aplicarla a los demás escenarios evalúa transferencia de esa configuración, no una optimización separada por escenario. Con x0 fijo GD es determinista: cambiar solo la semilla no produce nuevas inicializaciones. Por eso las condiciones explícitas se usan en las demostraciones y las corridas estadísticas mantienen sus inicializaciones aleatorias.

El gradiente central requiere 4 evaluaciones en 2D y 6 en 3D. El número real de pasos también depende de las evaluaciones de Armijo y los criterios de parada; no puede deducirse una reducción exacta de pasos considerando solo el costo del gradiente. El umbral de éxito tampoco es el valor óptimo, que es cero en ambas funciones.

Los HTML guardados incluyen animaciones interactivas y fuentes. Para que se vean en el blog deben publicarse y embeberse allí; generarlos localmente no actualiza la página publicada.
